In [1]:
!pip install synapseml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.8/625.8 kB 17.3 MB/s eta 0:00:00


In [2]:
# ------------------------------
# 1. Imports
# ------------------------------
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, hour, unix_timestamp, when
from pyspark.ml.feature import VectorAssembler
from synapse.ml.isolationforest import IsolationForest

spark = SparkSession.builder \
    .appName("Taxi-IsolationForest") \
    .config("spark.jars.packages", "com.microsoft.azure:synapseml_2.12:0.11.1") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

# ------------------------------
# 2. Load NYC Taxi Dataset
# ------------------------------
df = spark.read.option("header", True) \
               .option("inferSchema", True) \
               .csv("/kaggle/input/nyc-yellow-taxi-trip-data/yellow_tripdata_2015-01.csv")

# ------------------------------
# 3. Feature Engineering
# ------------------------------
# Convert pickup and dropoff datetime to timestamps
df = df.withColumn("pickup_ts", unix_timestamp("tpep_pickup_datetime")) \
       .withColumn("dropoff_ts", unix_timestamp("tpep_dropoff_datetime"))

# Trip duration in seconds
df = df.withColumn("trip_duration_sec", col("dropoff_ts") - col("pickup_ts"))

# Pickup hour
df = df.withColumn("pickup_hour", hour("tpep_pickup_datetime"))

# Remove trips with zero or negative duration
df = df.withColumn(
    "trip_duration_sec",
    when(col("trip_duration_sec") <= 0, None).otherwise(col("trip_duration_sec"))
)

# Average speed in km/h
df = df.withColumn(
    "avg_speed_kmh",
    col("trip_distance") / (col("trip_duration_sec") / 3600.0)
)

# ------------------------------
# 4. Prepare Features for IForest
# ------------------------------
feature_cols = ["trip_distance", "fare_amount", "trip_duration_sec", "avg_speed_kmh", "pickup_hour"]
df_clean = df.dropna(subset=feature_cols)

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_feat = assembler.transform(df_clean)

# ------------------------------
# 5. Train Isolation Forest Model
# ------------------------------
iforest = IsolationForest(
    featuresCol="features",
    predictionCol="prediction",
    scoreCol="anomalyScore",
    contamination=0.01,  # 1% anomalies
    numEstimators=100
)

model = iforest.fit(df_feat)

# ------------------------------
# 6. Predict Anomalies
# ------------------------------
pred = model.transform(df_feat)

# ------------------------------
# 7. Show Top 200 Anomalies
# ------------------------------
top_anomalies = pred.orderBy(col("anomalyScore").desc()).limit(200)
top_anomalies.show(truncate=False)


:: loading settings :: url = jar:file:/usr/local/lib/python3.11/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
com.microsoft.azure#synapseml_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-11554203-1b92-4e78-98bc-04fbe1df6256;1.0
	confs: [default]
	found com.microsoft.azure#synapseml_2.12;0.11.1 in central
	found com.microsoft.azure#synapseml-core_2.12;0.11.1 in central
	found org.scalactic#scalactic_2.12;3.2.14 in central
	found org.scala-lang#scala-reflect;2.12.15 in central
	found io.spray#spray-json_2.12;1.3.5 in central
	found com.jcraft#jsch;0.1.54 in central
	found org.apache.httpcomponents.client5#httpclient5;5.1.3 in central
	found org.apache.httpcomponents.core5#httpcore5;5.1.3 in central
	found org.apache.httpcomponents.core5#httpcore5-h2;5.1.3 in central
	found org.slf4j#slf4j-api;1.7.25 in central
	found commons-codec#commons-codec;1.15 in central
	found org.apache.httpcomponents#httpmime;4.5.13 in central
	found org.apache.httpcomponents#ht

+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+----------+----------+-----------------+-----------+------------------+--------------------------------------------+------------------+----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|pickup_longitude  |pickup_latitude   |RateCodeID|store_and_fwd_flag|dropoff_longitude |dropoff_latitude  |payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|pickup_ts |dropoff_ts|trip_duration_sec|pickup_hour|avg_speed_kmh     |features                                    |anomalyScore      |prediction|
+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+

**A) Abnormal Fare vs Distance/Time (Potential Fraud)**

In [3]:
fraud_suspects = pred \
    .filter(col("anomalyScore") > 0.7) \
    .filter((col("fare_amount") / (col("trip_distance") + 0.0001)) > 10) \
    .orderBy(col("anomalyScore").desc())

fraud_suspects.select(
    "tpep_pickup_datetime",
    "trip_distance",
    "trip_duration_sec",
    "fare_amount",
    "avg_speed_kmh",
    "anomalyScore"
).show(50, truncate=False)


+--------------------+-------------+-----------------+-----------+---------------------+------------------+
|tpep_pickup_datetime|trip_distance|trip_duration_sec|fare_amount|avg_speed_kmh        |anomalyScore      |
+--------------------+-------------+-----------------+-----------+---------------------+------------------+
|2015-01-21 23:23:56 |0.0          |86099            |135.33     |0.0                  |0.7599813623735571|
|2015-01-17 05:54:37 |0.0          |1428001          |965.0      |0.0                  |0.7539591812382286|
|2015-01-12 22:02:06 |0.0          |76619            |150.0      |0.0                  |0.7536785126988019|
|2015-01-26 00:53:46 |18.4         |41               |281.5      |1615.6097560975609   |0.7533094923804211|
|2015-01-22 23:22:12 |6.0          |38378            |322.5      |0.5628224503621867   |0.7526384099013886|
|2015-01-21 10:26:10 |13.78        |16002            |148.5      |3.100112485939257    |0.7505773620785092|
|2015-01-05 00:00:00 |0.0   

**B) Illogical Speeds (Too Fast or Too Slow)**

In [4]:
speed_anomalies = pred \
    .filter(col("anomalyScore") > 0.7) \
    .filter(
        (col("avg_speed_kmh") > 140) |   # extremely fast
        (col("avg_speed_kmh") < 2)       # extremely slow or stuck
    ) \
    .orderBy(col("anomalyScore").desc())

fraud_suspects.select(
    "tpep_pickup_datetime",
    "trip_distance",
    "trip_duration_sec",
    "fare_amount",
    "avg_speed_kmh",
    "anomalyScore"
).show(50, truncate=False)


+--------------------+-------------+-----------------+-----------+---------------------+------------------+
|tpep_pickup_datetime|trip_distance|trip_duration_sec|fare_amount|avg_speed_kmh        |anomalyScore      |
+--------------------+-------------+-----------------+-----------+---------------------+------------------+
|2015-01-21 23:23:56 |0.0          |86099            |135.33     |0.0                  |0.7599813623735571|
|2015-01-17 05:54:37 |0.0          |1428001          |965.0      |0.0                  |0.7539591812382286|
|2015-01-12 22:02:06 |0.0          |76619            |150.0      |0.0                  |0.7536785126988019|
|2015-01-26 00:53:46 |18.4         |41               |281.5      |1615.6097560975609   |0.7533094923804211|
|2015-01-22 23:22:12 |6.0          |38378            |322.5      |0.5628224503621867   |0.7526384099013886|
|2015-01-21 10:26:10 |13.78        |16002            |148.5      |3.100112485939257    |0.7505773620785092|
|2015-01-05 00:00:00 |0.0   